# Daily Challenge: Interactive Data Visualization with Matplotlib and Seaborn
### US Superstore Dataset

## 1. Data Preparation

In [ ]:
import io, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.widgets import CheckButtons, RadioButtons, Slider
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, SelectMultiple, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

# ── Load dataset ──────────────────────────────────────────────
URL = ("https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/"
       "Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls")
try:
    r  = requests.get(URL, timeout=15)
    df = pd.read_excel(io.BytesIO(r.content), engine='xlrd')
    print("Loaded from URL.")
except Exception as e:
    print(f"URL failed ({e}). Trying local file.")
    df = pd.read_excel('US Superstore data.xls', engine='xlrd')

print(f"Shape: {df.shape}")
df.head()

### 1.1 Data Cleaning and Preprocessing

In [ ]:
# Check for duplicates and missing values
print("Duplicate rows  :", df.duplicated().sum())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Fix date types
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Fill Postal Code if missing
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

# Derived time columns
df['Order Year']    = df['Order Date'].dt.year
df['Order Month']   = df['Order Date'].dt.month
df['Order YM']      = df['Order Date'].dt.to_period('M')
df['Order Quarter'] = df['Order Date'].dt.to_period('Q')

# Derived financial columns
df['Profit Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print("Preprocessing complete.")
print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Rows       : {len(df):,}")
df[['Sales','Profit','Discount','Quantity']].describe().round(2)

## 2. Data Visualization with Matplotlib

### 2.1 Interactive Line Chart — Sales Trends Over the Years

In [ ]:
# Prepare aggregations
monthly_all = (
    df.groupby(['Order YM', 'Category'])
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
    .reset_index()
)
monthly_all['Date'] = monthly_all['Order YM'].dt.to_timestamp()

yearly_cat = (
    df.groupby(['Order Year','Category'])
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
    .reset_index()
)

segment_monthly = (
    df.groupby(['Order YM','Segment'])['Sales']
    .sum().reset_index()
)
segment_monthly['Date'] = segment_monthly['Order YM'].dt.to_timestamp()

categories = df['Category'].unique().tolist()
segments   = df['Segment'].unique().tolist()
years      = sorted(df['Order Year'].unique())

print("Data prepared for interactive charts.")

In [ ]:
# ── Interactive widget: Sales trend with selectable granularity, metric and category ──
palette_cat = {'Furniture':'#4C72B0','Office Supplies':'#DD8452','Technology':'#55A868'}
palette_seg = {'Consumer':'#C44E52','Corporate':'#8172B2','Home Office':'#937860'}

def plot_sales_trend(granularity='Monthly', metric='Sales',
                     group_by='Category', show_grid=True):

    fig, ax = plt.subplots(figsize=(13, 6))

    if group_by == 'Category':
        groups   = categories
        palette  = palette_cat
        grp_col  = 'Category'
        data_src = monthly_all if granularity == 'Monthly' else yearly_cat
        date_col = 'Date' if granularity == 'Monthly' else 'Order Year'
    else:
        groups   = segments
        palette  = palette_seg
        grp_col  = 'Segment'
        seg_yr   = df.groupby(['Order Year','Segment'])[metric].sum().reset_index()
        data_src = segment_monthly if granularity == 'Monthly' else seg_yr
        date_col = 'Date' if granularity == 'Monthly' else 'Order Year'

    for grp in groups:
        sub = data_src[data_src[grp_col] == grp]
        ax.plot(sub[date_col], sub[metric],
                marker='o', markersize=3, linewidth=2,
                color=palette.get(grp), label=grp)

    # Shade year boundaries
    if granularity == 'Monthly':
        for yr in years:
            ax.axvline(pd.Timestamp(f'{yr}-01-01'),
                       color='grey', linewidth=0.7, linestyle='--', alpha=0.5)
            ax.text(pd.Timestamp(f'{yr}-02-01'),
                    ax.get_ylim()[1] * 0.97 if ax.get_ylim()[1] != 1 else 0.97,
                    str(yr), fontsize=8, color='grey', va='top')

    ax.set_title(f'{granularity} {metric} Trend by {group_by}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Period')
    ax.set_ylabel(f'{metric} ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
    if show_grid:
        ax.grid(True, alpha=0.35)
    ax.legend(title=group_by, loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

interact(
    plot_sales_trend,
    granularity = Dropdown(options=['Monthly','Yearly'], value='Monthly',
                           description='Granularity:'),
    metric      = Dropdown(options=['Sales','Profit'], value='Sales',
                           description='Metric:'),
    group_by    = Dropdown(options=['Category','Segment'], value='Category',
                           description='Group by:'),
    show_grid   = widgets.Checkbox(value=True, description='Show grid')
);

In [ ]:
# ── Static overview: year-over-year growth ──
yearly = df.groupby('Order Year')[['Sales','Profit']].sum().reset_index()
yearly['Sales_Growth']  = yearly['Sales'].pct_change() * 100
yearly['Profit_Growth'] = yearly['Profit'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(yearly['Order Year'], yearly['Sales'],
            color=['#4C72B0','#5585C3','#6698D6','#77ABE9'], edgecolor='white', width=0.6)
for i, (_, row) in enumerate(yearly.iterrows()):
    axes[0].text(row['Order Year'], row['Sales'] + 5000, f'${row["Sales"]/1e3:.0f}K',
                 ha='center', fontsize=9, fontweight='bold')
    if not np.isnan(row['Sales_Growth']):
        sign = '+' if row['Sales_Growth'] > 0 else ''
        axes[0].text(row['Order Year'], row['Sales'] / 2,
                     f'{sign}{row["Sales_Growth"]:.1f}%',
                     ha='center', fontsize=8, color='white', fontweight='bold')
axes[0].set_title('Annual Sales with YoY Growth', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(yearly['Order Year'], yearly['Profit'], marker='D', linewidth=2.5,
             color='#55A868', markersize=8, markeredgecolor='white', markeredgewidth=1.5)
for _, row in yearly.iterrows():
    axes[1].annotate(f'${row["Profit"]/1e3:.0f}K',
                     xy=(row['Order Year'], row['Profit']),
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Annual Profit Trend', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Profit ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].grid(True, alpha=0.3)

plt.suptitle('Year-over-Year Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Interactive Chart — Sales Distribution by State / Region

In [ ]:
# Note: A true geographic map requires geopandas + shapefiles.
# We build a highly interactive bar-chart map alternative with region drill-down.

state_agg = (
    df.groupby(['State','Region'])
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'),
         Orders=('Order ID','nunique'))
    .assign(Margin=lambda x: (x['Profit']/x['Sales']*100).round(1))
    .reset_index()
)
regions = ['All'] + sorted(df['Region'].unique().tolist())

def plot_state_sales(region='All', metric='Sales', top_n=15, color_by='Margin'):
    data = state_agg if region == 'All' else state_agg[state_agg['Region'] == region]
    data = data.sort_values(metric, ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(12, max(5, top_n * 0.45)))
    data_plot = data.sort_values(metric)

    if color_by == 'Margin':
        norm   = plt.Normalize(data_plot['Margin'].min(), data_plot['Margin'].max())
        colors = plt.cm.RdYlGn(norm(data_plot['Margin']))
    else:
        region_colors = {'Central':'#4C72B0','East':'#DD8452','South':'#55A868','West':'#C44E52'}
        colors = [region_colors.get(r, '#999') for r in data_plot['Region']]

    bars = ax.barh(data_plot['State'], data_plot[metric], color=colors, edgecolor='white')

    for bar in bars:
        w = bar.get_width()
        ax.text(w + data_plot[metric].max()*0.01, bar.get_y()+bar.get_height()/2,
                f'${w/1e3:.0f}K', va='center', fontsize=8)

    title_region = region if region != 'All' else 'All Regions'
    ax.set_title(f'Top {top_n} States by {metric} — {title_region}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(f'{metric} ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
    ax.grid(axis='x', alpha=0.3)

    if color_by == 'Margin':
        sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
        plt.colorbar(sm, ax=ax, label='Profit Margin (%)', shrink=0.6)
    else:
        from matplotlib.patches import Patch
        region_colors_full = {'Central':'#4C72B0','East':'#DD8452','South':'#55A868','West':'#C44E52'}
        ax.legend(handles=[Patch(color=c, label=r) for r,c in region_colors_full.items()],
                  title='Region', fontsize=8)

    plt.tight_layout()
    plt.show()

    summary = data_plot[['State','Region',metric,'Margin']].tail(5)[::-1]
    print(f"Top 5 of selection:")
    print(summary.to_string(index=False))

interact(
    plot_state_sales,
    region   = Dropdown(options=regions, value='All', description='Region:'),
    metric   = Dropdown(options=['Sales','Profit'], value='Sales', description='Metric:'),
    top_n    = IntSlider(min=5, max=49, value=15, description='Top N:'),
    color_by = Dropdown(options=['Margin','Region'], value='Margin', description='Color by:')
);

In [ ]:
# ── Regional summary heatmap ──
region_cat = (
    df.groupby(['Region','Category'])
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ['Sales','Profit']):
    pivot = region_cat.pivot(index='Region', columns='Category', values=col)
    sns.heatmap(pivot/1e3, annot=True, fmt='.0f', cmap='Blues' if col=='Sales' else 'RdYlGn',
                linewidths=0.5, ax=ax, center=0 if col=='Profit' else None)
    ax.set_title(f'{col} by Region & Category ($K)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Category')
    ax.set_ylabel('Region')

plt.suptitle('Regional Performance Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Visualization with Seaborn

### 3.1 Top 10 Products by Sales

In [ ]:
top10_products = (
    df.groupby(['Product Name','Category'])
    .agg(Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum'))
    .reset_index()
    .sort_values('Total_Sales', ascending=False)
    .head(10)
)

# Shorten long product names
top10_products['Short_Name'] = top10_products['Product Name'].apply(
    lambda x: x[:40]+'…' if len(x) > 40 else x
)

cat_palette = {'Furniture':'#4C72B0','Office Supplies':'#DD8452','Technology':'#55A868'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Seaborn horizontal bar: Sales ---
top10_sorted = top10_products.sort_values('Total_Sales')
sns.barplot(
    data=top10_sorted, x='Total_Sales', y='Short_Name',
    hue='Category', palette=cat_palette,
    dodge=False, legend=False, ax=axes[0]
)
for i, (_, row) in enumerate(top10_sorted.iterrows()):
    axes[0].text(row['Total_Sales'] + 500, i, f'${row["Total_Sales"]:,.0f}',
                 va='center', fontsize=8.5)
axes[0].set_title('Top 10 Products by Total Sales\n(Seaborn)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].set_ylabel('')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].grid(axis='x', alpha=0.35)

# Legend for category
from matplotlib.patches import Patch
axes[0].legend(
    handles=[Patch(color=c, label=k) for k,c in cat_palette.items()],
    title='Category', fontsize=8, loc='lower right'
)

# --- Paired bar: Sales vs Profit for top 10 ---
top10_melt = top10_products.melt(
    id_vars='Short_Name', value_vars=['Total_Sales','Total_Profit'],
    var_name='Metric', value_name='Value'
)
sns.barplot(
    data=top10_melt, x='Value', y='Short_Name',
    hue='Metric',
    palette={'Total_Sales':'#4C72B0','Total_Profit':'#55A868'},
    order=top10_sorted['Short_Name'],
    ax=axes[1]
)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 10 Products — Sales vs Profit\n(Seaborn)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Value ($)')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].legend(title='Metric', fontsize=9,
               labels=['Total Sales','Total Profit'])
axes[1].grid(axis='x', alpha=0.35)

plt.suptitle('Top 10 Products Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 10 products:")
print(top10_products[['Short_Name','Category','Total_Sales','Total_Profit']].to_string(index=False))

### 3.2 Scatter Plot — Profit vs Discount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Main scatter: Discount vs Profit, hue = Category ---
sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', palette=cat_palette,
    alpha=0.45, s=35, edgecolor='none', ax=axes[0]
)
sns.regplot(
    data=df, x='Discount', y='Profit',
    scatter=False, color='black',
    line_kws={'linewidth':2, 'linestyle':'--'},
    ax=axes[0]
)
axes[0].axhline(0, color='red', linewidth=1, alpha=0.5)
axes[0].text(0.55, 50, 'Break-even', fontsize=9, color='red', alpha=0.7)
axes[0].set_title('Discount vs Profit by Category\n(Seaborn scatterplot + regplot)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('Profit ($)')
axes[0].legend(title='Category', fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Faceted: one regression per category ---
for cat, color in cat_palette.items():
    sub = df[df['Category'] == cat]
    axes[1].scatter(sub['Discount'], sub['Profit'], color=color, alpha=0.3, s=25, label=cat)
    # regression line per category
    from numpy.polynomial.polynomial import polyfit
    x_range = np.linspace(sub['Discount'].min(), sub['Discount'].max(), 100)
    m, b = np.polyfit(sub['Discount'], sub['Profit'], 1)
    axes[1].plot(x_range, m * x_range + b, color=color, linewidth=2.5)

axes[1].axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
axes[1].set_title('Per-Category Regression Lines\n(Discount vs Profit)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Profit ($)')
axes[1].legend(title='Category', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impact of Discount on Profit', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical summary by discount band
df['Discount_Band'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0.0, 0.10, 0.20, 0.30, 0.50, 1.0],
    labels=['No discount','0–10%','10–20%','20–30%','30–50%','50%+']
)

print("Average Profit by Discount Band:")
print(
    df.groupby('Discount_Band', observed=True)['Profit']
    .agg(['mean','count','sum'])
    .rename(columns={'mean':'Avg Profit','count':'Transactions','sum':'Total Profit'})
    .round(2)
)

In [ ]:
# ── Extra Seaborn plot: Profit distribution by Discount Band and Category ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.boxplot(
    data=df, x='Discount_Band', y='Profit',
    hue='Category', palette=cat_palette,
    linewidth=1.2, ax=axes[0]
)
axes[0].axhline(0, color='red', linewidth=1, linestyle='--', alpha=0.6)
axes[0].set_title('Profit Distribution by Discount Band and Category',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Discount Band')
axes[0].set_ylabel('Profit ($)')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Category', fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

sns.violinplot(
    data=df[df['Discount'] > 0], x='Category', y='Profit',
    hue='Category', palette=cat_palette,
    inner='quartile', legend=False, ax=axes[1]
)
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6)
axes[1].set_title('Profit Distribution by Category\n(Discounted transactions only)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Profit ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Deeper Discount Analysis with Seaborn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Comparative Analysis — Matplotlib vs Seaborn

In [ ]:
# ── Side-by-side equivalent chart: same data, two libraries ──
sub_profit = (
    df.groupby('Sub-Category')['Profit']
    .sum().sort_values(ascending=False)
    .reset_index()
)
colors_sp = ['#55A868' if v >= 0 else '#E84040' for v in sub_profit['Profit']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Matplotlib version ──
axes[0].barh(sub_profit['Sub-Category'][::-1], sub_profit['Profit'][::-1],
             color=colors_sp[::-1], edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Profit by Sub-Category\n[Matplotlib — manual styling]',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Profit ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0].grid(axis='x', alpha=0.3)

# ── Seaborn version ──
sns.barplot(
    data=sub_profit, x='Profit', y='Sub-Category',
    palette=colors_sp, legend=False,
    order=sub_profit['Sub-Category'], ax=axes[1]
)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit by Sub-Category\n[Seaborn — concise styling]',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].set_ylabel('')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Matplotlib vs Seaborn — Same Chart, Two Approaches',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Comparative Observations: Matplotlib vs Seaborn

| Dimension | Matplotlib | Seaborn |
|---|---|---|
| **Lines of code** | More verbose — every visual element needs explicit configuration | Concise — a single function call handles grouping, colors and legend |
| **Interactivity** | Native integration with `ipywidgets` for dynamic filtering and sliders | No built-in interactivity; requires Matplotlib or Plotly underneath |
| **Statistical overlays** | Requires manual calculation (e.g., `np.polyfit` for regression) | Built-in `regplot`, confidence intervals, KDE via `histplot` |
| **Categorical data** | Straightforward but requires manual grouping and color mapping | Excellent: `hue`, `style`, `size` parameters handle grouping automatically |
| **Aesthetic defaults** | Plain white/grey; heavy manual styling needed for publication quality | Professional themes out of the box (`whitegrid`, `darkgrid`, `ticks`) |
| **Customisation ceiling** | Unlimited — any pixel-level tweak is possible | High but bounded — deep customisation falls back to Matplotlib axes |
| **Best use case** | Interactive dashboards, bespoke layouts, ipywidgets integration | Exploratory data analysis, statistical charts, stakeholder presentations |

**Recommendation:**  
Use **Matplotlib** when building interactive widgets or needing precise layout control. Use **Seaborn** for rapid statistical exploration and clean, publication-ready output. In practice the two are complementary: Seaborn plots sit on Matplotlib axes, so both libraries can be mixed in the same figure.

## 5. Code Quality and Key Insights

In [ ]:
# ── Final summary dashboard ──
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Annual sales bars
axes[0,0].bar(yearly['Order Year'], yearly['Sales'],
              color='#4C72B0', edgecolor='white', width=0.6)
axes[0,0].set_title('Annual Sales', fontweight='bold')
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0,0].grid(axis='y', alpha=0.3)

# 2. Category sales pie
cat_sales = df.groupby('Category')['Sales'].sum()
axes[0,1].pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
              colors=['#4C72B0','#DD8452','#55A868'],
              startangle=90, wedgeprops={'edgecolor':'white'})
axes[0,1].set_title('Sales by Category', fontweight='bold')

# 3. Top 10 states
t10 = state_agg.sort_values('Sales', ascending=True).tail(10)
axes[0,2].barh(t10['State'], t10['Sales'],
               color='#4C72B0', edgecolor='white')
axes[0,2].set_title('Top 10 States by Sales', fontweight='bold')
axes[0,2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[0,2].grid(axis='x', alpha=0.3)

# 4. Discount vs profit scatter
for cat, color in cat_palette.items():
    sub = df[df['Category'] == cat]
    axes[1,0].scatter(sub['Discount'], sub['Profit'],
                      color=color, alpha=0.3, s=15, label=cat)
axes[1,0].axhline(0, color='red', linewidth=1, linestyle='--')
axes[1,0].set_title('Discount vs Profit', fontweight='bold')
axes[1,0].set_xlabel('Discount')
axes[1,0].set_ylabel('Profit ($)')
axes[1,0].legend(fontsize=7)
axes[1,0].grid(True, alpha=0.3)

# 5. Sub-category profit
sp = sub_profit.sort_values('Profit')
colors_sub = ['#E84040' if v < 0 else '#55A868' for v in sp['Profit']]
axes[1,1].barh(sp['Sub-Category'], sp['Profit'], color=colors_sub, edgecolor='white')
axes[1,1].axvline(0, color='black', linewidth=0.8)
axes[1,1].set_title('Profit by Sub-Category', fontweight='bold')
axes[1,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1,1].grid(axis='x', alpha=0.3)

# 6. Monthly sales trend
m_total = df.groupby('Order YM')['Sales'].sum().reset_index()
m_total['Date'] = m_total['Order YM'].dt.to_timestamp()
axes[1,2].plot(m_total['Date'], m_total['Sales'], color='steelblue', linewidth=1.5)
axes[1,2].fill_between(m_total['Date'], m_total['Sales'], alpha=0.15, color='steelblue')
axes[1,2].set_title('Monthly Sales Trend', fontweight='bold')
axes[1,2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
axes[1,2].tick_params(axis='x', rotation=30, labelsize=7)
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('US Superstore — Executive Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Printed summary of key findings ──
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
margin       = total_profit / total_sales * 100
top_state    = state_agg.sort_values('Sales', ascending=False).iloc[0]
top_product  = top10_products.iloc[0]
loss_disc    = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100

print("=" * 56)
print("   KEY INSIGHTS — US SUPERSTORE DATA")
print("=" * 56)
print(f"  Total Revenue           : ${total_sales:>12,.0f}")
print(f"  Total Profit            : ${total_profit:>12,.0f}")
print(f"  Overall Profit Margin   : {margin:>11.1f}%")
print()
print(f"  Top State (Sales)       : {top_state['State']} (${top_state['Sales']:,.0f})")
print(f"  Top Product (Sales)     : {top_product['Short_Name']}")
print()
print(f"  Loss rate >20% discount : {loss_disc:.1f}% of transactions")
print(f"  Loss sub-categories     : Tables, Bookcases (Furniture)")
print()
print("  RECOMMENDATIONS:")
print("  1. Cap discounts at 20% to protect profitability")
print("  2. Focus marketing on California, NY and Texas")
print("  3. Review Furniture pricing — margin well below average")
print("  4. Invest in Q4 campaigns to leverage seasonal peaks")